# M1 · Maladies des feuilles d'olivier — entraînement, évaluation, export

Diagnostic par photo : **healthy** (saine), **peacock_spot** (œil de paon), **aculus_olearius** (acarien).
Dataset : Uğuz & Uysal, déjà téléchargé — **on ne le re-télécharge pas**.

Pipeline de ce notebook : splits 70/15/15 → EDA → baseline (petit CNN) → **MobileNetV3** avec
fine-tuning en 2 phases → évaluation (accuracy, F1 par classe, matrices de confusion, **Grad-CAM**)
→ **calibration** (temperature scaling) → **seuil de confiance 0.70** → export **TorchScript / ONNX / TFLite**
vers `models/m1/`.

**Google Colab** : `Exécution → Modifier le type d'exécution → T4 GPU`, et placez dans votre Drive :
- le dataset sous `ZitounAI/Olive_Leaf_Disease_Datasets_After_Preprocessing/` (copié depuis `Olive_Leaf_Disease_Datasets_After_Preprocessing/`),
- ce notebook et ses compagnons (`m1_common.py`, `prepare_splits.py`) sous `ZitounAI/notebooks/m1_disease/`.

**Local (PC, CPU possible)** : exécutez depuis `notebooks/m1_disease/` et réduisez les epochs dans la cellule de configuration.

In [ ]:
# ── Configuration : chemins, seed, hyperparamètres ──────────────────────────────
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SEED = 42
BATCH_SIZE = 32
EPOCHS_BASELINE = 8
EPOCHS_PHASE1 = 5       # MobileNetV3 : tête seule
EPOCHS_PHASE2 = 15      # MobileNetV3 : fine-tuning complet (CPU : réduisez, ex. 2 + 4)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE = Path("/content/drive/MyDrive/ZitounAI/notebooks/m1_disease")
    DATA_SRC = Path("/content/drive/MyDrive/ZitounAI/Olive_Leaf_Disease_Datasets_After_Preprocessing")
    DATA_DIR = Path("/content/data/m1_disease")   # splits sur le disque local de Colab (I/O rapides)
    OUT_DIR = Path("/content/models_m1")          # export, puis zip téléchargeable
else:
    HERE = Path.cwd()
    DATA_SRC = Path("C:/Users/LENOVO/.zcode/workspace/default/Olive_Leaf_Disease_Datasets_After_Preprocessing")
    DATA_DIR = HERE.parents[1] / "data" / "processed" / "m1_disease"
    OUT_DIR = HERE.parents[1] / "models" / "m1"

sys.path.insert(0, str(HERE))   # pour importer m1_common
FIG_DIR = HERE / "outputs"; FIG_DIR.mkdir(exist_ok=True)
print("Environnement :", "Colab" if IN_COLAB else "Local")
print("Dataset  :", DATA_SRC, "(existe :", DATA_SRC.exists(), ")")
print("Splits   :", DATA_DIR)
print("Export   :", OUT_DIR)

In [ ]:
# ── Dépendances (torch/torchvision/matplotlib sont déjà dans Colab) ─────────────
import importlib.util

def _pip_install_if_missing(module, pip_name=None):
    if importlib.util.find_spec(module) is None:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)

_pip_install_if_missing("matplotlib")
print("Dépendances de base OK")

In [ ]:
# ── Imports, seed fixe, matériel ────────────────────────────────────────────────
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

import m1_common as mc

mc.set_seed(SEED)
DEVICE = mc.get_device()
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU seulement"
print(f"torch {torch.__version__} | device : {DEVICE} | {gpu}")
print("Classes :", mc.CLASSES)

In [ ]:
# ── Splits 70/15/15 stratifiés PAR FEUILLE (anti-fuite des variantes Roboflow) ──
# Le script détecte seul les dossiers de classes (« Healthy », « Aculus Olive », « olivepeacockspot »)
# et regroupe les variantes d'une même feuille dans le même split. Seed fixe = reproductible.
if not (DATA_DIR / "train").is_dir():
    subprocess.run([sys.executable, str(HERE / "prepare_splits.py"),
                    "--src", str(DATA_SRC), "--dest", str(DATA_DIR), "--seed", str(SEED)], check=True)
else:
    print("Splits déjà présents :", DATA_DIR)

## 1 · Exploration des données

In [ ]:
# ── EDA : effectifs par split/classe, puis 9 feuilles d'exemple ─────────────────
train_dl, val_dl, test_dl = mc.make_loaders(DATA_DIR, batch_size=BATCH_SIZE, num_workers=2)
counts = pd.DataFrame({s: mc.LeafDataset(DATA_DIR / s, train=False).count_per_class()
                       for s in ["train", "val", "test"]}).reindex(mc.CLASSES).fillna(0).astype(int)
display(counts)
ax = counts.T.plot.bar(figsize=(7, 3.5), rot=0)
ax.set_title("Effectifs par classe et par split"); ax.set_ylabel("images")
plt.tight_layout(); plt.savefig(FIG_DIR / "effectifs.png", dpi=130); plt.show()

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for row, class_name in enumerate(mc.CLASSES):
    exemples = sorted((DATA_DIR / "train" / class_name).glob("*"))
    for col in range(3):
        axes[row, col].imshow(Image.open(exemples[SEED + col * 7]))
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(class_name, rotation=0, ha="right", fontsize=10)
plt.suptitle("Exemples du jeu d'entraînement"); plt.tight_layout(); plt.show()

In [ ]:
# ── Démonstration des augmentations « photo prise au champ » ────────────────────
mean = torch.tensor(mc.IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(mc.IMAGENET_STD).view(3, 1, 1)
augmente = mc.build_transforms(train=True)

def vers_affichage(tenseur):
    return (tenseur * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

demo = Image.open(sorted((DATA_DIR / "train" / "peacock_spot").glob("*"))[0]).convert("RGB")
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes[0, 0].imshow(demo); axes[0, 0].set_title("originale", fontsize=9)
for i, axis in enumerate(axes.flat[1:]):
    axis.imshow(vers_affichage(augmente(demo)))
    axis.set_title(f"augmentée #{i + 1}", fontsize=9)
for axis in axes.flat:
    axis.axis("off")
plt.suptitle("Luminosité, contraste, flou, rotation, recadrage, découpes — comme au champ")
plt.tight_layout(); plt.show()

In [ ]:
# ── Poids de classe (déséquilibre : peacock_spot ≈ 47% du dataset) ──────────────
class_weights = mc.class_weights_from(train_dl.dataset)
print("Pondération de la CrossEntropyLoss :",
      {c: round(w.item(), 3) for c, w in zip(mc.CLASSES, class_weights)})

## 2 · Baseline — petit CNN (aucun transfert)

In [ ]:
mc.set_seed(SEED)
baseline = mc.SmallCNN().to(DEVICE)
print(f"{sum(p.numel() for p in baseline.parameters()):,} paramètres")
baseline, _ = mc.train_model(
    baseline, train_dl, val_dl,
    epochs=EPOCHS_BASELINE, lr=1e-3, class_weights=class_weights, patience=3, device=DEVICE,
)

In [ ]:
results = {}
probs_b, labels_test = mc.evaluate(baseline, test_dl, DEVICE)
results["Baseline · petit CNN"] = mc.classification_report_dict(probs_b, labels_test, mc.CLASSES)
display(pd.DataFrame(results).T)
mc.plot_confusion(probs_b.argmax(1), labels_test, mc.CLASSES,
                  FIG_DIR / "confusion_baseline.png", "Baseline · petit CNN — matrice de confusion (test)")

## 3 · Modèle principal — MobileNetV3-Large, fine-tuning en 2 phases

- **Phase 1** : backbone ImageNet gelé, on n'entraîne que la tête (le classifieur) — lr élevé.
- **Phase 2** : backbone dégelé, lr faible pour ne pas détruire les features ImageNet.

In [ ]:
mc.set_seed(SEED)
mobilenet = mc.build_mobilenet(pretrained=True).to(DEVICE)

# Phase 1/2 : tête seule.
mc.set_backbone_frozen(mobilenet, frozen=True)
print(f"Phase 1 : {sum(p.numel() for p in mobilenet.parameters() if p.requires_grad):,} paramètres entraînables")
mobilenet, _ = mc.train_model(
    mobilenet, train_dl, val_dl,
    epochs=EPOCHS_PHASE1, lr=3e-3, class_weights=class_weights, patience=3, device=DEVICE,
)

In [ ]:
# Phase 2/2 : backbone dégelé, learning rate faible.
mc.set_backbone_frozen(mobilenet, frozen=False)
print(f"Phase 2 : {sum(p.numel() for p in mobilenet.parameters() if p.requires_grad):,} paramètres entraînables")
mobilenet, _ = mc.train_model(
    mobilenet, train_dl, val_dl,
    epochs=EPOCHS_PHASE2, lr=1e-4, class_weights=class_weights, patience=4, device=DEVICE,
)

## 4 · Évaluation finale — baseline vs MobileNetV3

In [ ]:
probs_m, labels_m = mc.evaluate(mobilenet, test_dl, DEVICE)
assert torch.equal(labels_test, labels_m), "même loader sans shuffle : même ordre attendu"
results["MobileNetV3 (2 phases)"] = mc.classification_report_dict(probs_m, labels_test, mc.CLASSES)
display(pd.DataFrame(results).T.round(4))
pd.DataFrame(results).T.round(4).to_csv(FIG_DIR / "comparaison_modeles.csv", encoding="utf-8")
mc.plot_confusion(probs_m.argmax(1), labels_m, mc.CLASSES,
                  FIG_DIR / "confusion_mobilenet.png", "MobileNetV3 — matrice de confusion (test)")

## 5 · Grad-CAM — la zone qui a fait décider le modèle (6 feuilles de test)

In [ ]:
gradcam = mc.GradCAM(mobilenet)   # cible : features[-1], la dernière couche de convolution
val_tf = mc.build_transforms(train=False)

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for row, class_name in enumerate(mc.CLASSES):
    exemples = sorted((DATA_DIR / "test" / class_name).glob("*"))[SEED:SEED + 2]
    for k, chemin in enumerate(exemples):
        image = Image.open(chemin).convert("RGB")
        batch = val_tf(image).unsqueeze(0).to(DEVICE)
        cam, index = gradcam(batch)
        axes[row, 2 * k].imshow(image)
        axes[row, 2 * k].set_title(f"{class_name} → prédit : {mc.CLASSES[index]}", fontsize=9)
        axes[row, 2 * k + 1].imshow(mc.GradCAM.overlay(image, cam))
        axes[row, 2 * k + 1].set_title("Grad-CAM", fontsize=9)
for axis in axes.flat:
    axis.axis("off")
plt.suptitle("Grad-CAM — rouge = zone utilisée pour la décision", y=1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / "gradcam_exemples.png", dpi=140, bbox_inches="tight"); plt.show()

## 6 · Calibration — temperature scaling

La température **T** est apprise sur le jeu de **validation** (minimisation de la NLL), puis appliquée partout (API incluse). L'argmax ne change pas : l'accuracy est identique, seules les confiances deviennent honnêtes.

In [ ]:
val_logits, val_labels = mc.collect_logits(mobilenet, val_dl, DEVICE)
test_logits, test_labels = mc.collect_logits(mobilenet, test_dl, DEVICE)
T = mc.find_temperature(val_logits, val_labels)

probs_raw = torch.softmax(test_logits, dim=1)
probs_cal = mc.calibrated_probs(test_logits, T)
print(f"Température retenue : T = {T:.3f}")
print(f"ECE avant calibration : {mc.expected_calibration_error(probs_raw, test_labels):.4f}")
print(f"ECE après calibration : {mc.expected_calibration_error(probs_cal, test_labels):.4f}")
assert torch.equal(probs_raw.argmax(1), probs_cal.argmax(1)), "l'argmain ne doit pas changer"
print("→ accuracy inchangée, confiances recalibrées (T est sauvegardée dans calibration.json)")

In [ ]:
def diagramme_fiabilite(probs, labels, axis, titre):
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).float().numpy()
    bornes = np.linspace(0, 1, 11)
    centres, precisions = [], []
    for bas, haut in zip(bornes[:-1], bornes[1:]):
        masque = ((conf > bas) & (conf <= haut)).numpy()
        if masque.any():
            centres.append((bas + haut) / 2)
            precisions.append(correct[masque].mean())
    axis.bar(centres, precisions, width=0.08, edgecolor="k", label="précision mesurée")
    axis.plot([0, 1], [0, 1], "k--", label="confiance parfaite")
    axis.set_xlabel("confiance annoncée"); axis.set_ylabel("précision réelle")
    axis.set_title(titre); axis.legend(fontsize=8); axis.set_xlim(0, 1); axis.set_ylim(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
diagramme_fiabilite(probs_raw, test_labels, axes[0], "Avant calibration")
diagramme_fiabilite(probs_cal, test_labels, axes[1], f"Après calibration (T = {T:.2f})")
plt.tight_layout(); plt.savefig(FIG_DIR / "diagramme_fiabilite.png", dpi=130); plt.show()

## 7 · Seuil de confiance 0.70 — « photo peu claire, reprenez-la »

En production : si la probabilité calibrée maximale est inférieure à 0.70, l'API ne renvoie pas de maladie ; le champ `advice` contient le message de reprise de photo et `lowConfidence` vaut `true` (voir `treatments.json`).

In [ ]:
conf_cal = probs_cal.max(dim=1).values
pred_cal = probs_cal.argmax(dim=1)
gardees = conf_cal >= mc.LOW_CONFIDENCE_THRESHOLD
print(f"Photos refusées (confiance < {mc.LOW_CONFIDENCE_THRESHOLD:.0%}) : {int((~gardees).sum())} / {len(gardees)}")
print(f"Accuracy sur les photos acceptées : {(pred_cal[gardees] == test_labels[gardees]).float().mean():.4f}"
      f"   (sans filtrage : {(pred_cal == test_labels).float().mean():.4f})")
if int((~gardees).sum()) > 0:
    print(f"Accuracy sur les photos refusées : {(pred_cal[~gardees] == test_labels[~gardees]).float().mean():.4f}"
          "  ← le modèle se trompe surtout quand il n'est pas sûr")
print("\nMessage renvoyé à l'agriculteur dans ce cas :\n ", mc.LOW_CONFIDENCE_ADVICE_FR)

## 8 · Export — TorchScript (API) · ONNX · TFLite (mobile) · configurations

In [ ]:
_pip_install_if_missing("onnx2tf", "onnx2tf")   # entraîne aussi TensorFlow si absent
metadata = {
    "dataset": "Uğuz & Uysal — Olive Leaf Disease Datasets (après prétraitement)",
    "backbone": "mobilenet_v3_large (IMAGENET1K_V2), fine-tuning en 2 phases",
    "metrics_baseline_test": results["Baseline · petit CNN"],
    "metrics_test": results["MobileNetV3 (2 phases)"],
    "torch_version": torch.__version__,
}
exportes = mc.export_all(mobilenet, OUT_DIR, val_logits, val_labels, mc.CLASSES, metadata)
for nom, chemin in exportes.items():
    print(f"  {nom:16s} {chemin}  ({chemin.stat().st_size / 1e6:.1f} Mo)")

if IN_COLAB:
    import shutil
    shutil.make_archive("/content/models_m1", "zip", OUT_DIR)
    print("\nZip prêt : /content/models_m1.zip — panneau Fichiers → télécharger,")
    print("puis côté PC : dézipper dans models/m1/ et `python scripts/models.py push m1`")

In [ ]:
# ── MLflow (optionnel) : trace l'expérience si MLFLOW_TRACKING_URI est défini ──
import os
if os.getenv("MLFLOW_TRACKING_URI"):
    try:
        import mlflow
        with mlflow.start_run(run_name="m1-mobilenetv3"):
            mlflow.log_params({"seed": SEED, "batch_size": BATCH_SIZE, "backbone": "mobilenet_v3_large",
                               "epochs_phase1": EPOCHS_PHASE1, "epochs_phase2": EPOCHS_PHASE2})
            mlflow.log_metrics(results["MobileNetV3 (2 phases)"])
            mlflow.log_artifact(str(OUT_DIR / "model.pt"))
        print("Run MLflow enregistré")
    except Exception as e:
        print("MLflow indisponible :", e)
else:
    print("MLflow désactivé (définissez la variable MLFLOW_TRACKING_URI pour l'activer)")

## Ensuite

1. **Partager les poids avec l'équipe** (Hugging Face, voir `models/README.md`) :
   déposez le contenu de l'export dans `models/m1/` puis `python scripts/models.py push m1`.
2. **Lancer l'API** : `docker compose up -d m1-disease` ou, en local,
   `cd ai-services/m1-disease && uvicorn app.main:app --port 8001` — sans modèle le service
   répond en mode mock, avec `models/m1/model.pt` il fait de vraies prédictions.
3. **Tester dans l'application** : page diagnostic du frontend → photo → maladie + confiance + conseil.
4. **Évaluer sur photos terrain** : rangez vos photos dans `notebooks/m1_disease/field_test/<classe>/`
   puis lancez `python evaluate_field.py`.

Le notebook doit s'exécuter de haut en bas sur un noyau frais ; ne committez pas les sorties (CONTRIBUTING.md).